# MvDEC standalone View2 pretraining screen (b_lr1e-4)

Controlled diagnostic. Pretrains only the S1 true-bottleneck View2 on REUTERS (seed 42, no View1, no refinement) and
scores K-Means on its latent. Grid: epochs {200, 1000} x latent {10, 20} at learning rate 0.0001.
Paper View2: 0.6501 / 0.5129. Pretrain-only control from earlier runs: about 0.515 / 0.32. Kaggle: GPU on, Internet on.

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle GPU before running this diagnostic."

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

PINNED_COMMIT = "554bdcd97ea1ad6fd77619c6104c2deb3be0258b"
REPO = Path("/kaggle/temp/mvdec_fpmax")
RESULTS = Path("/kaggle/working/mvdec-view2-screen-b_lr1e-4")
for path in (REPO, RESULTS):
    if path.exists():
        shutil.rmtree(path)
RESULTS.mkdir(parents=True)
subprocess.run(["git", "clone", "-q", "-b", "claude/mvdec-view2-screen",
                "https://github.com/vdat-18/mvdec_fpmax.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "-q", PINNED_COMMIT], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == PINNED_COMMIT
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"], check=True)
subprocess.run([sys.executable, "scripts/fetch_dekm_release.py", "--datasets", "REUTERS"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_mvdec_view2_screen.py"], cwd=REPO, check=True)

In [ ]:
configs = [f"epochs={e},lr=0.0001,latent={h}" for e in (200, 1000) for h in (10, 20)]
command = [sys.executable, "-u", "scripts/mvdec_view2_screen.py", "--output", str(RESULTS / "screen.json")]
for config in configs:
    command += ["--config", config]
with (RESULTS / "screen.log").open("w", encoding="utf-8") as log:
    result = subprocess.run(command, cwd=REPO, stdout=log, stderr=subprocess.STDOUT, check=False)
print("exit code", result.returncode)

In [ ]:
import json

data = json.loads((RESULTS / "screen.json").read_text())
print("paper View2 0.6501 / 0.5129")
for row in data["runs"]:
    print(f"epochs={row['epochs']} lr={row['lr']} latent={row['latent']}: "
          f"acc {row['view2_acc']:.4f} nmi {row['view2_nmi']:.4f} loss {row['final_loss']:.5f} ({row['seconds']} s)")
shutil.make_archive("/kaggle/working/mvdec-view2-screen-b_lr1e-4", "zip", RESULTS)